# Expérience — Estimation du prix par RAG filtré, version 2

Ce notebook reprend la stratégie de [la version 1](01_rag_filtre.ipynb) : retrouver des ventes comparables dans l'entraînement, puis demander à un LLM d'estimer le prix à partir de ces comparables. La partition, le modèle d'embedding, le LLM, le *system prompt*, la sortie structurée et l'évaluation restent identiques. Seules la recherche des comparables et la description des maisons changent.

### Résultats de la version 1 (292 maisons de validation)

| Méthode | RMSE des logarithmes | MAE |
|---|---|---|
| Médiane des comparables, RAG filtré v1 | 0,248 | 30 742 $ |
| **LLM + RAG filtré v1** | **0,140** | **18 168 $** |

L'analyse des erreurs de la version 1 a révélé trois faiblesses :

1. **Des informations décisives manquaient dans le texte.** Les deux plus grandes erreurs (maisons 31 et 813, surestimées de 75 % à 110 %) sont des maisons en zonage commercial sans air climatisé central (`MSZoning = C (all)`, `CentralAir = N`), deux champs absents de la description.
2. **Le filtre abandonnait le quartier au niveau 2.** Les 47 maisons évaluées à ce niveau avaient une RMSE des logarithmes de 0,199 avec le LLM, contre 0,124 au niveau 1.
3. **`MSSubClass` était un premier critère trop restrictif**, alors que `OverallQual` est la caractéristique la plus liée au prix ([étape 3](../03_analyse_exploratoire.ipynb), section 6).

### Changements de la version 2

| Élément | Version 1 | Version 2 |
|---|---|---|
| Premier critère du filtre | même `MSSubClass` | `OverallQual` à ±1 |
| Niveaux de filtre | 4, quartier abandonné au niveau 2 | 6, quartier conservé tant que possible |
| Métadonnées | 5 champs | 11 champs, dont `OverallQual`, `TotalSF` et `MSZoning` |
| Texte du prompt | 19 champs, codes bruts | 33 champs, valeurs lisibles, unités et variables calculées |
| Texte encodé | identique au prompt | version compacte des mêmes champs, sans troncature |
| Prix des comparables | prix total | prix total et prix au pi² habitable |
| Comparables dans le prompt | 5 | 5 (inchangé) |

**Coût.** Comme en version 1, la section 5 appelle l'API une fois par maison, soit 292 appels. Les prompts sont environ deux fois plus longs. Les réponses sont conservées dans un cache distinct de celui de la version 1.

## 1. Configuration

Les paramètres sont ceux de la version 1. Le cache des réponses et la collection ChromaDB sont distincts, car les textes et les métadonnées ont changé. Le cache de la version 1 est seulement relu, pour ajouter ses prédictions au tableau de comparaison de la section 6.

La clé de l'API doit être définie dans `OPENAI_API_KEY` ou dans le fichier `.env` à la racine du dépôt. Sans clé, seules les méthodes sans LLM sont évaluées.

In [ ]:
import hashlib
import json
import os
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Literal

import chromadb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from IPython.display import display
from openai import OpenAI
from pydantic import BaseModel, Field
from sentence_transformers import SentenceTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Repérer la racine depuis le dossier courant ou l'un de ses parents.
current = Path.cwd().resolve()
root = next(
    (path for path in (current, *current.parents) if (path / "pyproject.toml").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Ouvrir ce notebook depuis le dépôt MAP6014.")
data_dir = root / "data"

# Partition
RANDOM_STATE = 42
VALIDATION_SIZE = 0.20

# Recherche
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
K = 5                # comparables injectés dans le prompt
QUALITY_TOLERANCE = 1  # OverallQual de la cible ± 1
AREA_TOLERANCE = 0.25  # GrLivArea de la cible ± 25 %

# LLM
LLM_MODEL = "gpt-6-luna"
REASONING_EFFORT = "medium"
MAX_WORKERS = 8      # appels simultanés à l'API
N_LLM = None         # None : toute la validation; un entier (ex. 20) pour un essai
RUN_NAIVE_LLM = False  # True : évaluer aussi le LLM avec un RAG naïf (double le coût)

# Fichiers locaux, ignorés par Git car situés dans data/
chroma_dir = data_dir / "chroma_ames"
cache_path = data_dir / "cache_rag" / "llm_predictions_v2.jsonl"
v1_cache_path = data_dir / "cache_rag" / "llm_predictions.jsonl"  # lecture seule

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100


def format_dollars(value):
    """Afficher un montant en dollars avec une espace comme séparateur de milliers."""
    return f"{value:,.0f} $".replace(",", " ")

## 2. Partition entraînement / validation

Nous séparons `train.csv` en **80 % pour l'entraînement** et **20 % pour la validation**, avec une graine fixe. Seules les maisons d'entraînement sont indexées : les maisons de validation ne peuvent donc jamais se retrouver parmi leurs propres comparables, et leur prix n'est jamais montré au modèle. `test.csv`, qui ne contient pas de prix, sera utilisé plus tard pour une soumission Kaggle.

In [ ]:
read_options = {"na_values": ["NA"], "keep_default_na": False}
train_full = pd.read_csv(data_dir / "train.csv", **read_options)
test = pd.read_csv(data_dir / "test.csv", **read_options)
assert train_full.shape == (1460, 81) and test.shape == (1459, 80)

train_part, valid_part = train_test_split(
    train_full, test_size=VALIDATION_SIZE, random_state=RANDOM_STATE
)
train_part = train_part.reset_index(drop=True)
valid_part = valid_part.reset_index(drop=True)
assert not set(train_part["Id"]) & set(valid_part["Id"])

display(pd.DataFrame({
    "Partition": ["Entraînement (indexé)", "Validation (évaluée)", "test.csv (plus tard)"],
    "Maisons": [len(train_part), len(valid_part), len(test)],
    "Prix médian": [format_dollars(train_part["SalePrice"].median()),
                    format_dollars(valid_part["SalePrice"].median()), "inconnu"],
}))

Les deux partitions ont des prix médians proches, ce qui indique que la séparation aléatoire n'a pas créé de déséquilibre important. La partition est identique à celle de la version 1 : les résultats des deux versions portent sur les mêmes 292 maisons.

## 3. Description textuelle et indexation

### Texte d'une propriété

La version 2 enrichit la description de chaque maison :

- **Champs ajoutés**, choisis d'après l'analyse exploratoire et les erreurs de la version 1 : zonage (`MSZoning`), proximité d'une route ou d'une voie ferrée (`Condition1`), air climatisé central, fonctionnalité, état extérieur, qualité et surface aménagée du sous-sol, surface de l'étage, demi-salles de bain, surface du garage et mois de vente.
- **Variables calculées** : âge de la maison à la vente, années depuis la rénovation et surface totale (`GrLivArea + TotalBsmtSF`).
- **Valeurs lisibles** : les codes sont remplacés par leur libellé de `data_description.txt` (`TA` → `Typical/Average`, `C (all)` → `Commercial`, `IDOTRR` → `Iowa DOT and Rail Road`), les scores sont notés sur 10 et les surfaces portent leur unité. Les codes des CSV qui diffèrent du dictionnaire (étape 3, section 2) sont harmonisés.
- **Absence d'équipement** : pour `BsmtQual`, une valeur `NA` devient `No Basement`, le libellé du dictionnaire, au lieu d'être omise.

Les noms de variables sont conservés devant chaque valeur, car le *system prompt* y fait référence. **Le prix n'en fait jamais partie.**

In [ ]:
# Libellés lisibles des codes, extraits de data_description.txt.
description_text = (data_dir / "data_description.txt").read_text(encoding="utf-8")
code_labels, current_variable = {}, None
for line in description_text.splitlines():
    header = re.match(r"^(\w+):", line)
    if header:
        current_variable = header.group(1)
        code_labels[current_variable] = {}
        continue
    entry = re.match(r"^\s+(\S[^\t]*?)\s*\t+\s*(.*\S)", line)
    if current_variable and entry:
        code_labels[current_variable][entry.group(1).strip()] = entry.group(2).strip()

# Codes écrits autrement dans les CSV que dans le dictionnaire (étape 3, section 2).
code_labels["MSZoning"]["C (all)"] = code_labels["MSZoning"]["C"]
code_labels["Neighborhood"]["NAmes"] = code_labels["Neighborhood"]["Names"]
code_labels["BldgType"].update({
    "2fmCon": code_labels["BldgType"]["2FmCon"],
    "Duplex": code_labels["BldgType"]["Duplx"],
    "Twnhs": code_labels["BldgType"]["TwnhsI"],
})

MONTHS = ["January", "February", "March", "April", "May", "June", "July", "August",
          "September", "October", "November", "December"]

# Champs du texte : (nom affiché, fonction qui renvoie la valeur lisible ou None).
def label(column):
    def read(row):
        value = row[column]
        if pd.isna(value):
            return code_labels[column].get("NA")  # « No Basement », etc.
        return code_labels[column].get(str(value), str(value))
    return read


def score(column):
    return lambda row: f"{int(row[column])}/10 ({code_labels[column][str(int(row[column]))]})"


def number(column, unit=""):
    def read(row):
        value = row[column]
        if pd.isna(value):
            return None
        return f"{value:,.0f}{unit}"
    return read


TEXT_FIELDS = [
    ("Neighborhood", label("Neighborhood")),
    ("MSZoning", label("MSZoning")),
    ("Condition1", label("Condition1")),
    ("BldgType", label("BldgType")),
    ("HouseStyle", label("HouseStyle")),
    ("OverallQual", score("OverallQual")),
    ("OverallCond", score("OverallCond")),
    ("YearBuilt", lambda row: str(int(row["YearBuilt"]))),
    ("YearRemodAdd", lambda row: str(int(row["YearRemodAdd"]))),
    ("AgeAtSale", lambda row: f"{int(row['YrSold'] - row['YearBuilt'])} years"),
    ("YearsSinceRemodel", lambda row: f"{max(int(row['YrSold'] - row['YearRemodAdd']), 0)} years"),
    ("GrLivArea", number("GrLivArea", " sq ft")),
    ("2ndFlrSF", number("2ndFlrSF", " sq ft")),
    ("TotalBsmtSF", number("TotalBsmtSF", " sq ft")),
    ("BsmtFinSF1", number("BsmtFinSF1", " sq ft")),
    ("BsmtQual", label("BsmtQual")),
    ("TotalSF", lambda row: None if pd.isna(row["TotalBsmtSF"])
        else f"{row['GrLivArea'] + row['TotalBsmtSF']:,.0f} sq ft"),
    ("LotArea", number("LotArea", " sq ft")),
    ("GarageCars", number("GarageCars")),
    ("GarageArea", number("GarageArea", " sq ft")),
    ("FullBath", number("FullBath")),
    ("HalfBath", number("HalfBath")),
    ("BedroomAbvGr", number("BedroomAbvGr")),
    ("KitchenQual", label("KitchenQual")),
    ("ExterQual", label("ExterQual")),
    ("ExterCond", label("ExterCond")),
    ("CentralAir", label("CentralAir")),
    ("Functional", label("Functional")),
    ("Fireplaces", number("Fireplaces")),
    ("PoolArea", number("PoolArea", " sq ft")),
    ("SaleCondition", label("SaleCondition")),
    ("MoSold", lambda row: MONTHS[int(row["MoSold"]) - 1]),
    ("YrSold", lambda row: str(int(row["YrSold"]))),
]


def build_property_text(row):
    """Décrire une maison par « Variable=valeur lisible », sans valeurs absentes."""
    parts = []
    for name, read in TEXT_FIELDS:
        value = read(row)
        if value is not None:
            parts.append(f"{name}={value}")
    return "; ".join(parts)


# Texte encodé par MiniLM : mêmes informations, codes bruts, sans unités.
# Le texte lisible dépasse la limite de 256 jetons du modèle pour la plupart des maisons;
# cette version compacte (environ 210 jetons) n'est jamais tronquée.
EMBEDDING_FIELDS = [
    "Neighborhood", "MSZoning", "Condition1", "BldgType", "HouseStyle", "OverallQual",
    "OverallCond", "YearBuilt", "YearRemodAdd", "AgeAtSale", "GrLivArea", "2ndFlrSF",
    "TotalBsmtSF", "BsmtFinSF1", "TotalSF", "BsmtQual", "LotArea", "GarageCars",
    "GarageArea", "FullBath", "HalfBath", "BedroomAbvGr", "KitchenQual", "ExterQual",
    "ExterCond", "CentralAir", "Functional", "Fireplaces", "PoolArea", "SaleCondition",
    "MoSold", "YrSold",
]


def build_embedding_text(row):
    values = row.copy()
    values["AgeAtSale"] = row["YrSold"] - row["YearBuilt"]
    values["TotalSF"] = row["GrLivArea"] + row["TotalBsmtSF"]
    parts = []
    for column in EMBEDDING_FIELDS:
        value = values[column]
        if pd.isna(value):
            value = "None"
        elif isinstance(value, float) and value.is_integer():
            value = int(value)
        parts.append(f"{column}={value}")
    return "; ".join(parts)


assert "SalePrice" not in {name for name, _ in TEXT_FIELDS} | set(EMBEDDING_FIELDS)
print(build_property_text(valid_part.iloc[0]))
print()
print(build_embedding_text(valid_part.iloc[0]))

### Indexation dans ChromaDB

Chaque maison est stockée avec trois éléments : le **document**, c'est-à-dire le texte lisible injecté dans le prompt; l'**embedding**, calculé sur le texte compact; et les **métadonnées** utilisées par les filtres. La version 2 ajoute aux métadonnées de la version 1 (`SalePrice`, `Neighborhood`, `BldgType`, `MSSubClass`, `GrLivArea`) six champs : `OverallQual`, `YearBuilt`, `TotalSF`, `MSZoning`, `SaleCondition` et `YrSold`. Seuls `OverallQual`, `Neighborhood` et `GrLivArea` servent aux filtres actuels; les autres permettront de tester d'autres filtres sans réindexer.

Le nom de la collection commence par `ames_v2_` et dépend des identifiants de l'entraînement. La collection de la version 1 n'est pas modifiée.

In [ ]:
encoder = SentenceTransformer(EMBEDDING_MODEL)
chroma_client = chromadb.PersistentClient(path=str(chroma_dir))

split_signature = hashlib.sha256(
    ",".join(map(str, sorted(train_part["Id"]))).encode()
).hexdigest()[:12]
collection_name = f"ames_v2_{split_signature}"
collection = chroma_client.get_or_create_collection(collection_name)

if collection.count() != len(train_part):
    chroma_client.delete_collection(collection_name)
    collection = chroma_client.create_collection(collection_name)
    documents = [build_property_text(row) for _, row in train_part.iterrows()]
    embeddings = encoder.encode(
        [build_embedding_text(row) for _, row in train_part.iterrows()],
        batch_size=64, normalize_embeddings=True,
    )
    collection.add(
        ids=train_part["Id"].astype(str).tolist(),
        documents=documents,
        embeddings=embeddings.tolist(),
        metadatas=[
            {
                "SalePrice": float(row["SalePrice"]),
                "Neighborhood": str(row["Neighborhood"]),
                "BldgType": str(row["BldgType"]),
                "MSSubClass": int(row["MSSubClass"]),
                "GrLivArea": float(row["GrLivArea"]),
                "OverallQual": int(row["OverallQual"]),
                "YearBuilt": int(row["YearBuilt"]),
                "TotalSF": float(row["GrLivArea"] + row["TotalBsmtSF"]),
                "MSZoning": str(row["MSZoning"]),
                "SaleCondition": str(row["SaleCondition"]),
                "YrSold": int(row["YrSold"]),
            }
            for _, row in train_part.iterrows()
        ],
    )
    print(f"Collection {collection_name} créée : {collection.count()} maisons indexées.")
else:
    print(f"Collection {collection_name} réutilisée : {collection.count()} maisons.")
assert collection.count() == len(train_part)

## 4. Recherche filtrée avec repli

La recherche garde le principe de la version 1 : la maison cible est encodée, puis ChromaDB renvoie les *k* = 5 voisins les plus proches parmi les maisons qui satisfont le premier filtre. S'il y en a moins de cinq, nous passons au niveau suivant. Les filtres de la version 2 portent d'abord sur ce que l'embedding représente mal, les **nombres** (qualité et surface), tout en conservant le **quartier** aussi longtemps que possible :

| Niveau | Filtre sur les métadonnées |
|---|---|
| 1 | `OverallQual` ±1, même `Neighborhood` et `GrLivArea` ±25 % |
| 2 | `OverallQual` ±1 et `GrLivArea` ±25 % |
| 3 | `OverallQual` ±1 et même `Neighborhood` |
| 4 | `OverallQual` ±1 |
| 5 | même `Neighborhood` |
| 6 | aucun filtre |

**Choix des niveaux.** Nous avons comparé plusieurs ordres de filtres sans LLM, en prédisant chaque maison de l'**entraînement** par la médiane de ses cinq comparables, la maison elle-même étant exclue de la recherche. La validation n'a pas servi à ce choix. Avec cette mesure, l'ordre retenu obtient une RMSE des logarithmes de 0,186, contre 0,224 pour la recherche naïve. Un ordre sans quartier au niveau 1 obtenait 0,181, mais ne gardait que 86 % de comparables du même quartier, contre 96 % pour l'ordre retenu. Nous avons privilégié le quartier, car en version 1 le LLM se trompait nettement plus lorsque ses comparables venaient d'un autre quartier.

Le RAG naïf utilise la même fonction, avec le seul niveau « aucun filtre ».

In [ ]:
def similar_quality(row):
    quality = int(row["OverallQual"])
    return [{"OverallQual": {"$gte": quality - QUALITY_TOLERANCE}},
            {"OverallQual": {"$lte": quality + QUALITY_TOLERANCE}}]


def similar_area(row):
    area = float(row["GrLivArea"])
    return [{"GrLivArea": {"$gte": area * (1 - AREA_TOLERANCE)}},
            {"GrLivArea": {"$lte": area * (1 + AREA_TOLERANCE)}}]


def same_neighborhood(row):
    return [{"Neighborhood": str(row["Neighborhood"])}]


def all_of(*groups):
    """Combiner des listes de conditions ChromaDB en un seul filtre where."""
    conditions = [condition for group in groups for condition in group]
    return conditions[0] if len(conditions) == 1 else {"$and": conditions}


FILTER_LEVELS = [
    ("Qualité ±1 + quartier + surface ±25 %",
     lambda row: all_of(similar_quality(row), same_neighborhood(row), similar_area(row))),
    ("Qualité ±1 + surface ±25 %",
     lambda row: all_of(similar_quality(row), similar_area(row))),
    ("Qualité ±1 + quartier",
     lambda row: all_of(similar_quality(row), same_neighborhood(row))),
    ("Qualité ±1", lambda row: all_of(similar_quality(row))),
    ("Quartier", lambda row: all_of(same_neighborhood(row))),
    ("Aucun filtre", lambda row: None),
]


def build_find_similar(collection, encoder, k=K, levels=FILTER_LEVELS):
    """Créer une fonction qui renvoie (comparables, niveau de filtre utilisé)."""

    def find_similar(row, exclude_id=None):
        query = encoder.encode([build_embedding_text(row)], normalize_embeddings=True)
        comparables, level_name = [], None
        for level_name, make_filter in levels:
            result = collection.query(
                query_embeddings=query.tolist(),
                n_results=k + 1,  # une de plus, au cas où exclude_id serait retourné
                where=make_filter(row),
                include=["documents", "metadatas", "distances"],
            )
            comparables = [
                {"Id": int(item_id), "text": document, "price": metadata["SalePrice"],
                 "GrLivArea": metadata["GrLivArea"],
                 "Neighborhood": metadata["Neighborhood"], "distance": distance}
                for item_id, document, metadata, distance in zip(
                    result["ids"][0], result["documents"][0],
                    result["metadatas"][0], result["distances"][0],
                )
                if item_id != str(exclude_id)
            ][:k]
            if len(comparables) >= k:
                break
        return comparables, level_name

    return find_similar


find_similar = build_find_similar(collection, encoder)
find_similar_naive = build_find_similar(
    collection, encoder, levels=[("Aucun filtre", lambda row: None)]
)


def retrieve_all(frame, search):
    """Chercher les comparables de chaque maison; une erreur donne une liste vide."""
    retrieved = {}
    for _, row in frame.iterrows():
        try:
            comparables, level_name = search(row)
        except Exception as error:  # le repli sans RAG sera utilisé
            comparables, level_name = [], f"Échec ({type(error).__name__})"
        retrieved[int(row["Id"])] = {"comparables": comparables, "level": level_name}
    return retrieved


retrieved_filtered = retrieve_all(valid_part, find_similar)
retrieved_naive = retrieve_all(valid_part, find_similar_naive)
assert all(len(item["comparables"]) == K for item in retrieved_filtered.values())

example = valid_part.iloc[0]
example_comparables = retrieved_filtered[int(example["Id"])]
print("Cible :", build_property_text(example))
print("Niveau de filtre :", example_comparables["level"])
display(pd.DataFrame(example_comparables["comparables"])[
    ["Id", "Neighborhood", "GrLivArea", "price", "distance"]
])

### Niveaux de filtre utilisés et qualité des comparables

Avant d'appeler le LLM, nous examinons gratuitement deux questions : à quel niveau de filtre la recherche s'arrête-t-elle, et les comparables trouvés sont-ils déjà de bons indicateurs du prix? Pour la seconde, nous utilisons une **référence sans LLM** : le prix médian des cinq comparables. Si le LLM n'améliore pas cette référence, il n'apporte rien de plus que la recherche.

In [ ]:
level_order = [name for name, _ in FILTER_LEVELS]
levels_used = pd.Series(
    [item["level"] for item in retrieved_filtered.values()], name="Niveau"
)
level_counts = levels_used.value_counts().reindex(
    level_order + sorted(set(levels_used) - set(level_order)), fill_value=0
)

# Part des comparables situés dans le même quartier que la maison cible.
valid_by_id = valid_part.set_index("Id")


def same_neighborhood_share(retrieved):
    shares = [
        np.mean([c["Neighborhood"] == valid_by_id.loc[house_id, "Neighborhood"]
                 for c in item["comparables"]])
        for house_id, item in retrieved.items() if item["comparables"]
    ]
    return float(np.mean(shares) * 100)


fig, ax = plt.subplots(figsize=(8, 3.5))
sns.barplot(x=level_counts.values, y=level_counts.index, color="tab:blue", ax=ax)
for position, count in enumerate(level_counts.values):
    ax.text(count + 2, position, f"{count} ({count / len(valid_part):.0%})", va="center")
ax.set_title(" Niveau de filtre où la recherche s'arrête (premier niveau donnant 5 comparables)")
ax.set_xlabel("Maisons de validation")
ax.set_ylabel("")
ax.set_xlim(0, level_counts.max() * 1.2)
plt.tight_layout()
plt.show()

print(f"Comparables du même quartier : {same_neighborhood_share(retrieved_filtered):.1f} % "
      f"(RAG filtré) contre {same_neighborhood_share(retrieved_naive):.1f} % (RAG naïf)")

**Question.** Les nouveaux filtres trouvent-ils assez de comparables sans abandonner le quartier?

**Observation.** Lors de notre exécution, le niveau 1 (qualité, quartier et surface) suffit pour **262 maisons sur 292 (90 %)**. Vingt-sept maisons (9 %) passent au niveau 2, sans contrainte de quartier, et seulement trois descendent plus bas; aucune n'atteint les niveaux 5 et 6. Au total, **94,2 %** des comparables proviennent du même quartier que la maison cible, contre 89,7 % avec le filtre de la version 1 et 93,8 % avec la recherche naïve de la version 2.

**Interprétation.** Les tolérances de ±1 pour la qualité et de ±25 % pour la surface laissent généralement au moins cinq maisons dans le même quartier : contraindre la qualité et la surface ne coûte presque rien en localisation. Les maisons du niveau 2 sont, par définition, celles dont le quartier compte moins de cinq maisons de qualité et de surface semblables.

**Conséquence.** Le LLM reçoit maintenant des comparables du même marché local et de taille et de qualité proches de la maison cible : il a moins d'ajustements à faire. La section 6 vérifie si cela réduit ses erreurs.

## 5. Prompt et LLM

### Prompts

Le *system prompt* et la structure du *user prompt* sont **identiques à la version 1**, afin que les écarts de résultats viennent de la recherche et de la description des maisons. Deux éléments changent dans le *user prompt* : chaque maison est décrite par le texte enrichi, et le prix de chaque comparable est accompagné de son **prix au pied carré habitable** (`GrLivArea`), une mesure qu'un évaluateur utilise pour comparer des maisons de tailles différentes.

In [ ]:
SYSTEM_PROMPT = """You are an expert residential real estate appraiser for Ames, Iowa, \
specialized in home sales from 2006 to 2010. Estimate the sale price in US dollars of the \
target property from its characteristics and, when provided, from comparable sales.

Key value factors:
- Location (Neighborhood) and dwelling type (BldgType, HouseStyle).
- Overall material and finish quality (OverallQual) and overall condition (OverallCond).
- Above-ground living area (GrLivArea), basement area (TotalBsmtSF) and lot area (LotArea).
- Age and renovation (YearBuilt, YearRemodAdd) relative to the sale year (YrSold).
- Garage capacity, bathrooms, bedrooms, fireplaces, kitchen and exterior quality.
- Sale condition: Partial (new construction) and Abnorml (foreclosure, short sale) \
sales often deviate from typical market prices.

Adjust the comparable prices for the differences with the target property instead of \
averaging them. Report your confidence as low, medium or high and briefly explain the \
main adjustments."""


def build_user_prompt(row, comparables):
    """Prompt avec comparables (RAG) ou, si la liste est vide, sans comparables."""
    target = build_property_text(row)
    if not comparables:
        return f"Estimate the price for this property:\n{target}"
    blocks = [
        f"Similar property {i}:\n{c['text']}\n"
        f"Price: ${c['price']:,.0f} (${c['price'] / c['GrLivArea']:,.0f} per sq ft of GrLivArea)"
        for i, c in enumerate(comparables, start=1)
    ]
    return "\n\n".join(blocks) + f"\n\nNow estimate the price for this property:\n{target}"


class PricePrediction(BaseModel):
    predicted_price: float = Field(gt=0, description="Estimated sale price in US dollars")
    confidence: Literal["low", "medium", "high"]
    reasoning: str = Field(description="Short justification of the main adjustments")


print(SYSTEM_PROMPT)
print("\n" + "-" * 80 + "\n")
print(build_user_prompt(example, example_comparables["comparables"]))

### Appels au modèle et cache

Chaque appel utilise l'API *Responses* d'OpenAI avec une **sortie structurée** : le SDK convertit la classe Pydantic `PricePrediction` en schéma JSON et renvoie un objet validé. Les réponses sont ajoutées au fichier `data/cache_rag/llm_predictions.jsonl`. La clé du cache combine le modèle, l'effort de raisonnement et les deux prompts : modifier l'un d'eux déclenche de nouveaux appels, tandis qu'une réexécution identique relit le cache.

Les appels sont faits en parallèle (`MAX_WORKERS`). Une erreur sur une maison (limite de débit persistante, réponse invalide) est enregistrée sans interrompre les autres; ces maisons sont exclues de l'évaluation et comptées dans le bilan.

Pendant l'exécution, une barre de progression affiche la RMSE des logarithmes cumulée et le nombre d'erreurs. Chaque maison terminée est aussi affichée par l'écart entre le prix prédit et le prix réel : en vert sous 10 %, en jaune sous 20 %, en rouge au-delà.

In [ ]:
load_dotenv(root / ".env")
LLM_AVAILABLE = bool(os.environ.get("OPENAI_API_KEY"))
if not LLM_AVAILABLE:
    print("OPENAI_API_KEY absente : les sections LLM sont ignorées, "
          "seules les méthodes sans LLM seront évaluées.")
else:
    llm_client = OpenAI(max_retries=5)

cache_path.parent.mkdir(parents=True, exist_ok=True)
cache = {}
if cache_path.exists():
    for line in cache_path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            entry = json.loads(line)
            cache[entry["key"]] = entry
cache_lock = threading.Lock()


def cache_key(user_prompt):
    payload = json.dumps([LLM_MODEL, REASONING_EFFORT, SYSTEM_PROMPT, user_prompt])
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def predict_price(house_id, row, comparables):
    """Prédire un prix avec le LLM; repli sans RAG si aucun comparable."""
    user_prompt = build_user_prompt(row, comparables)
    key = cache_key(user_prompt)
    if key in cache:
        return cache[key]

    start = time.perf_counter()
    response = llm_client.responses.parse(
        model=LLM_MODEL,
        reasoning={"effort": REASONING_EFFORT},
        instructions=SYSTEM_PROMPT,
        input=user_prompt,
        text_format=PricePrediction,
    )
    parsed = response.output_parsed
    if parsed is None:
        raise ValueError("Réponse sans sortie structurée (refus ou réponse incomplète).")
    entry = {
        "key": key,
        "Id": int(house_id),
        "with_rag": bool(comparables),
        **parsed.model_dump(),
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "seconds": round(time.perf_counter() - start, 2),
    }
    with cache_lock:
        cache[key] = entry
        with cache_path.open("a", encoding="utf-8") as file:
            file.write(json.dumps(entry, ensure_ascii=False) + "\n")
    return entry


GREEN, YELLOW, RED, RESET = "\033[92m", "\033[93m", "\033[91m", "\033[0m"


def gap_color(gap):
    """Vert : écart < 10 %; jaune : < 20 %; rouge : au-delà."""
    if abs(gap) < 0.10:
        return GREEN
    if abs(gap) < 0.20:
        return YELLOW
    return RED


def run_llm(frame, retrieved, label):
    """Prédire toutes les maisons de frame en parallèle; renvoie un DataFrame.

    Pendant l'exécution, une barre de progression indique la RMSE des logarithmes
    cumulée, et chaque maison terminée est affichée par son écart au prix réel.
    """
    rows, errors, squared_log_errors = [], [], []
    actual_prices = frame.set_index("Id")["SalePrice"]
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {
            pool.submit(predict_price, int(row["Id"]), row,
                        retrieved[int(row["Id"])]["comparables"]): int(row["Id"])
            for _, row in frame.iterrows()
        }
        progress = tqdm(as_completed(futures), total=len(futures), desc=label)
        for future in progress:
            house_id = futures[future]
            try:
                entry = future.result()
            except Exception as error:
                errors.append({"Id": house_id, "Erreur": f"{type(error).__name__}: {error}"})
                print(f"{RED}échec{RESET} ", end="")
            else:
                rows.append({**entry, "Id": house_id})
                ratio = entry["predicted_price"] / actual_prices[house_id]
                squared_log_errors.append(np.log(ratio) ** 2)
                print(f"{gap_color(ratio - 1)}{ratio - 1:+.0%}{RESET} ", end="")
            progress.set_postfix({
                "RMSE log": f"{np.sqrt(np.mean(squared_log_errors)):.3f}"
                if squared_log_errors else "-",
                "erreurs": len(errors),
            })
    print(f"\n{label} : {len(rows)} prédictions, {len(errors)} erreurs.")
    return pd.DataFrame(rows), pd.DataFrame(errors)

### Test sur une maison

Avant de lancer toute la validation, nous vérifions la chaîne complète sur **une seule maison** : recherche des comparables, prompt, appel au LLM et lecture de la réponse structurée. Ce test coûte un seul appel. Sa réponse est conservée dans le cache : la même maison ne sera pas facturée de nouveau lors de l'évaluation complète.

Pour tester une autre maison, changer `TEST_INDEX` (position dans la validation).

In [ ]:
TEST_INDEX = 0

if not LLM_AVAILABLE:
    print("OPENAI_API_KEY absente : test ignoré.")
else:
    test_row = valid_part.iloc[TEST_INDEX]
    test_id = int(test_row["Id"])
    test_retrieval = retrieved_filtered[test_id]
    test_entry = predict_price(test_id, test_row, test_retrieval["comparables"])

    actual = test_row["SalePrice"]
    predicted = test_entry["predicted_price"]
    comparables_median_price = np.median([c["price"] for c in test_retrieval["comparables"]])

    print("Maison :", build_property_text(test_row))
    print(f"Niveau de filtre : {test_retrieval['level']}\n")
    display(pd.DataFrame({
        "Prix": [actual, predicted, comparables_median_price],
        "Écart au prix réel": [0.0, predicted / actual - 1, comparables_median_price / actual - 1],
    }, index=["Prix réel", "LLM + RAG filtré", "Médiane des comparables"])
        .style.format({"Prix": format_dollars, "Écart au prix réel": "{:+.1%}"}))
    print(f"Erreur log : {abs(np.log(predicted) - np.log(actual)):.4f}")
    print(f"Confiance : {test_entry['confidence']}")
    print(f"Justification : {test_entry['reasoning']}\n")
    print(f"Jetons : {test_entry['input_tokens']:,} en entrée, "
          f"{test_entry['output_tokens']:,} en sortie | durée : {test_entry['seconds']} s")

In [ ]:
llm_frame = (
    valid_part if N_LLM is None
    else valid_part.sample(n=N_LLM, random_state=RANDOM_STATE)
)

llm_results, llm_errors = {}, {}
if LLM_AVAILABLE:
    strategies = {"LLM + RAG filtré": retrieved_filtered}
    if RUN_NAIVE_LLM:
        strategies["LLM + RAG naïf"] = retrieved_naive
    for label, retrieved in strategies.items():
        llm_results[label], llm_errors[label] = run_llm(llm_frame, retrieved, label)
        if not llm_errors[label].empty:
            display(llm_errors[label].head())

## 6. Évaluation

### Métriques

La métrique principale est celle de Kaggle, la **RMSE des logarithmes** (logarithme naturel), présentée à l'étape 2 :

$$
\operatorname{RMSE}_{\log}
=
\sqrt{\frac{1}{n}\sum_{i=1}^{n}
\left(\log(\widehat{y}_i)-\log(y_i)\right)^2}.
$$

Nous ajoutons la RMSE et la MAE en dollars ainsi que le R², calculé sur les prix en dollars. Toutes les méthodes sont comparées **sur les mêmes maisons** :

| Méthode | Description |
|---|---|
| Médiane globale | Prix médian de l'entraînement pour toutes les maisons : référence minimale. |
| Médiane des comparables, RAG naïf | Prix médian des 5 voisins trouvés sans filtre, sans LLM. |
| Médiane des comparables, RAG filtré | Prix médian des 5 voisins de la version 2, sans LLM. |
| LLM + RAG filtré (v1) | Prédictions de la version 1, relues dans son cache. |
| LLM + RAG filtré | Prix estimé par le LLM avec la recherche et le texte de la version 2. |

In [ ]:
def regression_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    return {
        "RMSE log": np.sqrt(np.mean((np.log(y_pred) - np.log(y_true)) ** 2)),
        "RMSE ($)": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE ($)": mean_absolute_error(y_true, y_pred),
        "R²": r2_score(y_true, y_pred),
        "Maisons": len(y_true),
    }


def comparables_median(retrieved):
    return pd.Series({
        house_id: np.median([c["price"] for c in item["comparables"]])
        if item["comparables"] else np.nan
        for house_id, item in retrieved.items()
    })


predictions = pd.DataFrame({
    "Id": valid_part["Id"],
    "SalePrice": valid_part["SalePrice"],
    "Niveau": valid_part["Id"].map(lambda house_id: retrieved_filtered[house_id]["level"]),
    "Médiane globale": train_part["SalePrice"].median(),
    "Médiane des comparables, RAG naïf": valid_part["Id"].map(comparables_median(retrieved_naive)),
    "Médiane des comparables, RAG filtré": valid_part["Id"].map(comparables_median(retrieved_filtered)),
})

# Prédictions de la version 1 (lecture seule de son cache).
if v1_cache_path.exists():
    v1_entries = [json.loads(line) for line in
                  v1_cache_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    v1_prices = (pd.DataFrame(v1_entries).query("with_rag")
                 .drop_duplicates("Id", keep="last").set_index("Id")["predicted_price"])
    predictions["LLM + RAG filtré (v1)"] = predictions["Id"].map(v1_prices)

for label, frame in llm_results.items():
    if not frame.empty:
        indexed = frame.set_index("Id")
        predictions[label] = predictions["Id"].map(indexed["predicted_price"])
        predictions[f"{label} — confiance"] = predictions["Id"].map(indexed["confidence"])

methods = [column for column in predictions.columns
           if column not in {"Id", "SalePrice", "Niveau"} and "confiance" not in column]
# Comparer toutes les méthodes sur les maisons pour lesquelles chacune a une prédiction.
common = predictions.dropna(subset=methods)

results = pd.DataFrame(
    {method: regression_metrics(common["SalePrice"], common[method]) for method in methods}
).T.sort_values("RMSE log")
display(results.style.format({
    "RMSE log": "{:.4f}", "RMSE ($)": "{:,.0f}", "MAE ($)": "{:,.0f}", "R²": "{:.3f}",
    "Maisons": "{:.0f}",
}))

**Interprétation des méthodes sans LLM.** Le tableau porte sur 291 maisons, car le cache de la version 1 contient une prédiction de moins que la validation. La médiane des comparables filtrés passe de 0,248 en version 1 à **0,192** en version 2, soit une baisse de 23 %. Elle devient nettement meilleure que la recherche naïve (0,238) : le filtre joue maintenant le rôle attendu. La recherche naïve est un peu moins bonne qu'en version 1 (0,229); l'embedding seul ne profite donc pas des champs ajoutés, et le gain vient des filtres.

**Interprétation du LLM.** À compléter après exécution : la version 2 améliore-t-elle la RMSE des logarithmes de 0,140 obtenue par la version 1? La médiane des comparables s'étant améliorée de 23 %, un gain plus faible pour le LLM indiquerait qu'il corrigeait déjà en version 1 une bonne partie des défauts de la recherche.

In [ ]:
squared_log_errors = pd.DataFrame({
    method: (np.log(common[method]) - np.log(common["SalePrice"])) ** 2
    for method in methods if method != "Médiane globale"
})
by_level = (
    squared_log_errors.groupby(common["Niveau"]).mean().pipe(np.sqrt)
    .reindex([level for level in level_order if level in set(common["Niveau"])])
    .round(3)
)
by_level.insert(0, "Maisons", common["Niveau"].value_counts())
by_level.columns = ["Maisons"] + [f"RMSE log — {name}" for name in by_level.columns[1:]]
display(by_level)

Le tableau détaille l'erreur selon le niveau de filtre de la version 2. Au niveau 1, la médiane des comparables filtrés atteint 0,182, contre 0,209 pour les comparables naïfs des mêmes maisons. L'écart est beaucoup plus grand au niveau 2 (27 maisons) : 0,209 contre 0,367. Pour ces maisons, dont la combinaison de qualité et de surface est rare dans leur quartier, la recherche naïve, sans contrainte de qualité ni de surface, se trompe beaucoup plus. La colonne de la version 1 sert de repère : sur ces mêmes groupes, le LLM de la version 1 obtenait 0,141 et 0,124. Les niveaux 3 et 4 ne comptent que trois maisons et ne permettent pas de conclure.

### Prédictions et prix réels

In [ ]:
panels = [
    "Médiane des comparables, RAG filtré", "LLM + RAG filtré (v1)",
] + [label for label in llm_results if label in common.columns]
panels = [method for method in panels if method in common.columns]
ticks = [50_000, 100_000, 200_000, 400_000, 700_000]
tick_labels = [f"{tick // 1000} k" for tick in ticks]

fig, axes = plt.subplots(1, len(panels), figsize=(5.5 * len(panels), 5.5), squeeze=False)
limits = (common["SalePrice"].min() * 0.8, common["SalePrice"].max() * 1.2)
for ax, method in zip(axes[0], panels):
    sns.scatterplot(x=common["SalePrice"], y=common[method], hue=common["Niveau"],
                    hue_order=level_order, alpha=0.7, s=30, ax=ax)
    ax.plot(limits, limits, color="black", linestyle="--", linewidth=1, label="Prédiction parfaite")
    ax.set(xscale="log", yscale="log", xlim=limits, ylim=limits,
           xlabel="Prix réel ($, échelle log)", ylabel="Prix prédit ($, échelle log)")
    ax.set_xticks(ticks, tick_labels)
    ax.set_yticks(ticks, tick_labels)
    ax.minorticks_off()
    rmse_log = regression_metrics(common["SalePrice"], common[method])["RMSE log"]
    ax.set_title(f"{method}\nRMSE log = {rmse_log:.4f}")
    ax.legend(title="Niveau de filtre (v2)", fontsize=7)
plt.tight_layout()
plt.show()

**Question.** Les prédictions de la version 2 suivent-elles mieux les prix réels que celles de la version 1, notamment aux extrémités?

**Observation et interprétation.** À compléter après exécution. Points à examiner : les maisons très abordables, surestimées en version 1 (en particulier les maisons 31 et 813, en zonage commercial); les maisons chères, sous-estimées; et les ventes `Partial`.

### Erreurs selon le niveau de filtre et la confiance déclarée

In [ ]:
llm_labels = [label for label in llm_results if label in common.columns]
if not llm_labels:
    print("Aucune prédiction du LLM : section ignorée.")
else:
    label = llm_labels[0]
    errors = common.assign(**{
        "Erreur absolue log": (np.log(common[label]) - np.log(common["SalePrice"])).abs(),
        "Confiance": common[f"{label} — confiance"],
    })
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    sns.boxplot(data=errors, x="Niveau", y="Erreur absolue log",
                order=[level for level in level_order if level in set(errors["Niveau"])],
                color="tab:orange", ax=axes[0])
    axes[0].set_title(f"{label} : erreur selon le niveau de filtre")
    axes[0].tick_params(axis="x", rotation=15)
    sns.boxplot(data=errors, x="Confiance", y="Erreur absolue log",
                order=[c for c in ["low", "medium", "high"] if c in set(errors["Confiance"])],
                color="tab:green", ax=axes[1])
    axes[1].set_title(f"{label} : erreur selon la confiance déclarée")
    plt.tight_layout()
    plt.show()

    display(errors.groupby("Niveau")["Erreur absolue log"].agg(["count", "median"]).round(3))
    display(errors.groupby("Confiance")["Erreur absolue log"].agg(["count", "median"]).round(3))

**Question.** Les erreurs sont-elles plus grandes lorsque la recherche doit relâcher le filtre, et la confiance annoncée par le modèle est-elle informative?

**Observation et interprétation.** À compléter après exécution. Une confiance utile devrait correspondre à des erreurs plus faibles pour `high` que pour `low`. Si ce n'est pas le cas, la confiance n'est pas calibrée et ne doit pas servir à pondérer les prédictions.

### Plus grandes erreurs

Le tableau suivant présente les dix maisons dont l'erreur du LLM est la plus grande, avec la justification fournie par le modèle. Ces cas aident à comprendre si les erreurs viennent de mauvais comparables, d'un mauvais ajustement ou d'une vente particulière (par exemple `Partial` ou `Abnorml`).

In [ ]:
if llm_labels:
    label = llm_labels[0]
    reasoning = llm_results[label].set_index("Id")["reasoning"]
    worst = (
        errors.assign(Écart=lambda frame: frame[label] / frame["SalePrice"] - 1)
        .sort_values("Erreur absolue log", ascending=False)
        .head(10)
    )
    worst = worst.merge(valid_part[["Id", "Neighborhood", "MSSubClass", "SaleCondition",
                                    "GrLivArea", "OverallQual"]], on="Id")
    worst["Justification"] = worst["Id"].map(reasoning)
    display(worst[["Id", "SalePrice", label, "Écart", "Niveau", "Confiance", "Neighborhood",
                   "MSSubClass", "SaleCondition", "GrLivArea", "OverallQual",
                   "Justification"]]
            .style.format({"SalePrice": "{:,.0f}", label: "{:,.0f}", "Écart": "{:+.0%}"}))

    usage = llm_results[label]
    print(f"Jetons d'entrée : {usage['input_tokens'].sum():,} | "
          f"jetons de sortie (raisonnement inclus) : {usage['output_tokens'].sum():,} | "
          f"durée médiane d'un appel : {usage['seconds'].median():.1f} s")

**Interprétation.** À compléter après exécution.

## 7. Limites et pistes

- **Effets combinés.** La version 2 modifie à la fois les filtres, les champs et la présentation des valeurs. Une amélioration globale ne dit pas lequel de ces changements en est responsable.
- **Choix des filtres avec une mesure sans LLM.** L'ordre des niveaux a été choisi d'après la médiane des comparables. Le meilleur ordre pour cette médiane n'est pas forcément le meilleur pour le LLM, qui ajuste les prix lui-même.
- **Contamination possible.** Le jeu de données d'Ames est public. Le LLM a peut-être vu ces ventes pendant son entraînement; des valeurs lisibles, proches des descriptions publiées, pourraient faciliter ce rappel.
- **Une seule partition de validation.** Avec 292 maisons, un écart de quelques millièmes de RMSE entre les versions 1 et 2 peut venir du hasard.
- **Résultats non déterministes.** Deux exécutions du même prompt peuvent donner des prix différents; la variabilité n'est pas mesurée.
- **Coût.** Les prompts de la version 2 sont environ deux fois plus longs que ceux de la version 1.
- **Prédiction de `test.csv`.** Pour la soumission Kaggle, il faudra indexer les 1 460 maisons de `train.csv`, puis prédire les 1 459 maisons de `test.csv` avec les mêmes paramètres. Le texte gère déjà les valeurs absentes de `test.csv`.

### Fiche de l'expérience

À remplir après chaque exécution complète : commit, date, `RANDOM_STATE`, `K`, `QUALITY_TOLERANCE`, `AREA_TOLERANCE`, modèle d'embedding, `LLM_MODEL`, `REASONING_EFFORT`, `N_LLM`, nombre d'erreurs, métriques du tableau de la section 6 et coût approximatif.

## Références

- [Expérience RAG filtré, version 1](01_rag_filtre.ipynb).
- [Étape 2 — Présentation du problème](../02_presentation_probleme.ipynb), pour la métrique de Kaggle.
- [Étape 3 — Analyse exploratoire](../03_analyse_exploratoire.ipynb), pour le choix des caractéristiques, des filtres et l'harmonisation des codes.
- [Description locale des variables](../../data/data_description.txt), source des libellés lisibles.
- [sentence-transformers/all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2), modèle d'embedding (256 jetons au maximum).
- [Documentation de ChromaDB](https://docs.trychroma.com/), pour les requêtes filtrées (`where`, `$gte`, `$lte`).
- [Documentation de l'API OpenAI — sorties structurées](https://platform.openai.com/docs/guides/structured-outputs).

**Convention de dépôt.** Les sorties de calcul sont effacées avant le commit, conformément au [guide de collaboration](../../docs/collaboration.md). L'index ChromaDB et les caches des réponses restent dans `data/`, qui n'est pas versionné.